# Neural Networks from Scratch

We build a neural network using **only NumPy** — no frameworks, no magic.

This is the single most important notebook in the deep learning track. Every framework (PyTorch, TensorFlow) automates what we'll do by hand here. Understanding these mechanics means you'll *debug* models instead of *guessing*.

**What we'll build:**
1. A single neuron (it's just logistic regression)
2. A multi-layer network with forward and backward passes
3. A complete `NeuralNetwork` class that solves XOR
4. Train on real-ish data and watch decision boundaries evolve

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. What Is a Neuron?

A neuron takes inputs, multiplies each by a weight, sums them up, adds a bias, and passes the result through an activation function.

```
inputs      weights
  x1 ──w1──┐
  x2 ──w2──┼──► Σ ──► activation ──► output
  x3 ──w3──┘
         + bias
```

Mathematically: `output = activation(w1*x1 + w2*x2 + w3*x3 + b)`

That's it. Every neuron in every network — from a simple classifier to GPT — follows this pattern. The only things that change are:
- How many neurons and how they're connected
- Which activation function
- How we set the weights (training)

---
## 2. Activation Functions

Without activation functions, stacking layers is pointless — multiple linear transforms collapse into a single linear transform. Activations introduce **non-linearity**, letting networks learn curved decision boundaries.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def tanh(z):
    return np.tanh(z)

def relu(z):
    return np.maximum(0, z)

z = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, fn, name, color in zip(axes,
                                [sigmoid, tanh, relu],
                                ['Sigmoid', 'Tanh', 'ReLU'],
                                ['#e74c3c', '#3498db', '#2ecc71']):
    ax.plot(z, fn(z), color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
    ax.set_title(name, fontsize=14, fontweight='bold')
    ax.set_xlabel('z')
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('activation(z)')
plt.tight_layout()
plt.show()

**Why ReLU is preferred:**

| Function | Range | Problem |
|----------|-------|---------|
| Sigmoid | (0, 1) | Gradients → 0 for large \|z\| (vanishing gradient) |
| Tanh | (-1, 1) | Same vanishing gradient issue |
| ReLU | [0, ∞) | Gradient is 1 for z > 0 → no vanishing gradient |

ReLU is computationally cheap and doesn't saturate for positive values. Its one weakness: "dead neurons" (gradient = 0 when z < 0), which variants like Leaky ReLU fix.

---
## 3. A Single Neuron — It's Just Logistic Regression

A single neuron with sigmoid activation IS logistic regression. Let's prove it.

In [ ]:
class SingleNeuron:
    def __init__(self, n_inputs):
        self.w = np.random.randn(n_inputs) * 0.01
        self.b = 0.0
    
    def forward(self, X):
        z = X @ self.w + self.b
        return sigmoid(z)
    
    def train(self, X, y, lr=0.1, epochs=100):
        losses = []
        for _ in range(epochs):
            y_hat = self.forward(X)
            loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))
            losses.append(loss)
            
            error = y_hat - y
            self.w -= lr * (X.T @ error) / len(y)
            self.b -= lr * np.mean(error)
        return losses

In [ ]:
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1.0])

neuron = SingleNeuron(2)
losses = neuron.train(X_and, y_and, lr=1.0, epochs=200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(losses, color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss — AND Gate')
ax1.grid(True, alpha=0.3)

predictions = neuron.forward(X_and)
colors = ['#e74c3c' if y == 0 else '#2ecc71' for y in y_and]
ax2.scatter(X_and[:, 0], X_and[:, 1], c=colors, s=200, edgecolors='black', zorder=5)
for i, (x, p) in enumerate(zip(X_and, predictions)):
    ax2.annotate(f'{p:.2f}', (x[0]+0.05, x[1]+0.05), fontsize=12)
ax2.set_title('Predictions (green=1, red=0)')
ax2.set_xlim(-0.3, 1.5)
ax2.set_ylim(-0.3, 1.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Predictions: {np.round(predictions, 2)}")
print(f"A single neuron solves AND — it draws one straight line to separate 0s from 1s.")

---
## 4. Multi-Layer Networks

A single neuron can only draw a straight line. For complex patterns, we **stack neurons into layers**.

```
Input Layer      Hidden Layer      Output Layer
  (2 neurons)     (4 neurons)       (1 neuron)

    x1 ──────►  h1 ──────┐
      \  ╱      h2 ──────┼──► output
       ╳        h3 ──────┤
      / ╲       h4 ──────┘
    x2 ──────►
```

Each layer transforms the data into a new representation. More layers = more complex transformations = more complex patterns the network can recognize.

**Key insight:** each hidden neuron learns a different "feature" of the input. The output layer combines these features to make the final decision.

---
## 5. Forward Pass — Matrix Multiplication at Each Layer

The forward pass is just repeated matrix multiplication + activation. Let's trace through with actual numbers.

In [ ]:
X = np.array([[0.5, 0.8]])

W1 = np.array([[0.2, -0.3, 0.5],
               [0.4,  0.1, -0.2]])
b1 = np.array([0.1, -0.1, 0.0])

W2 = np.array([[0.6],
               [-0.4],
               [0.3]])
b2 = np.array([0.05])

print("=== Layer 1 ===")
z1 = X @ W1 + b1
print(f"  z1 = X @ W1 + b1 = {z1}")
a1 = relu(z1)
print(f"  a1 = relu(z1)    = {a1}")

print("\n=== Layer 2 (output) ===")
z2 = a1 @ W2 + b2
print(f"  z2 = a1 @ W2 + b2 = {z2}")
a2 = sigmoid(z2)
print(f"  output = sigmoid(z2) = {a2}")

print(f"\nThe network takes input {X[0]} and outputs {a2[0][0]:.4f}")

**Dimensions check** — the most common bug in neural networks:

| Step | Shape | Why |
|------|-------|-----|
| X | (1, 2) | 1 sample, 2 features |
| W1 | (2, 3) | 2 inputs → 3 hidden neurons |
| z1 = X @ W1 | (1, 3) | 3 hidden activations |
| W2 | (3, 1) | 3 hidden → 1 output |
| z2 = a1 @ W2 | (1, 1) | 1 output |

---
## 6. Loss Functions — How Wrong Are We?

The loss function quantifies the gap between predictions and truth. Two workhorses:

- **MSE** (Mean Squared Error) → regression: \\(\frac{1}{n}\sum(y - \hat{y})^2\\)
- **Cross-entropy** → classification: \\(-\frac{1}{n}\sum[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]\\)

In [ ]:
y_true = 1.0
y_preds = np.linspace(0.01, 0.99, 100)

mse_losses = (y_true - y_preds) ** 2
ce_losses = -(y_true * np.log(y_preds) + (1 - y_true) * np.log(1 - y_preds))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(y_preds, mse_losses, color='#e74c3c', linewidth=2.5)
ax1.set_title('MSE Loss (y_true = 1)', fontweight='bold')
ax1.set_xlabel('Prediction')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(y_preds, ce_losses, color='#3498db', linewidth=2.5)
ax2.set_title('Cross-Entropy Loss (y_true = 1)', fontweight='bold')
ax2.set_xlabel('Prediction')
ax2.set_ylabel('Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Cross-entropy penalizes confident wrong predictions MUCH harder than MSE.")
print(f"Predict 0.01 when true=1: MSE={0.99**2:.3f}, CE={-np.log(0.01):.3f}")

---
## 7. Backpropagation — THE Key Algorithm

Training = adjusting weights to reduce loss. But HOW do we know which way to adjust each weight?

**Backpropagation** = chain rule applied systematically from output back to input.

For a weight w in layer l:

```
∂Loss/∂w = ∂Loss/∂output × ∂output/∂z × ∂z/∂w
```

The gradients "flow backward" through the network — that's why it's called back-propagation.

In [ ]:
x_val = 2.0
w_val = 0.5
b_val = 0.1
y_val = 1.0

z = w_val * x_val + b_val
a = sigmoid(z)
loss = -(y_val * np.log(a) + (1 - y_val) * np.log(1 - a))

print("=== Forward Pass ===")
print(f"  z = w*x + b = {w_val}*{x_val} + {b_val} = {z}")
print(f"  a = sigmoid(z) = sigmoid({z}) = {a:.6f}")
print(f"  loss = -[y*log(a) + (1-y)*log(1-a)] = {loss:.6f}")

print("\n=== Backward Pass (Chain Rule) ===")
dl_da = -(y_val / a - (1 - y_val) / (1 - a))
print(f"  ∂L/∂a = {dl_da:.6f}")

da_dz = a * (1 - a)
print(f"  ∂a/∂z = a*(1-a) = {da_dz:.6f}")

dz_dw = x_val
dz_db = 1.0
print(f"  ∂z/∂w = x = {dz_dw}")
print(f"  ∂z/∂b = 1")

dl_dw = dl_da * da_dz * dz_dw
dl_db = dl_da * da_dz * dz_db
print(f"\n  ∂L/∂w = ∂L/∂a × ∂a/∂z × ∂z/∂w = {dl_dw:.6f}")
print(f"  ∂L/∂b = ∂L/∂a × ∂a/∂z × ∂z/∂b = {dl_db:.6f}")

In [ ]:
eps = 1e-5

def compute_loss(w, b):
    z = w * x_val + b
    a = sigmoid(z)
    return -(y_val * np.log(a) + (1 - y_val) * np.log(1 - a))

numerical_dl_dw = (compute_loss(w_val + eps, b_val) - compute_loss(w_val - eps, b_val)) / (2 * eps)
numerical_dl_db = (compute_loss(w_val, b_val + eps) - compute_loss(w_val, b_val - eps)) / (2 * eps)

print("Verification — analytical vs numerical gradients:")
print(f"  ∂L/∂w: analytical={dl_dw:.6f}, numerical={numerical_dl_dw:.6f}")
print(f"  ∂L/∂b: analytical={dl_db:.6f}, numerical={numerical_dl_db:.6f}")
print(f"\n  Match? {np.allclose(dl_dw, numerical_dl_dw) and np.allclose(dl_db, numerical_dl_db)} ✓")

---
## 8. Training Loop — The Full Picture

Every neural network trains with the same loop:

```
for each epoch:
    1. Forward pass  → compute predictions
    2. Compute loss  → how wrong are we?
    3. Backward pass → compute gradients
    4. Update weights → w = w - learning_rate * gradient
```

Let's implement this for a 2-layer network from scratch.

In [ ]:
def sigmoid_derivative(a):
    return a * (1 - a)

def relu_derivative(z):
    return (z > 0).astype(float)

np.random.seed(42)
n_input, n_hidden, n_output = 2, 8, 1

W1 = np.random.randn(n_input, n_hidden) * np.sqrt(2.0 / n_input)
b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, n_output) * np.sqrt(2.0 / n_hidden)
b2 = np.zeros((1, n_output))

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)

lr = 0.5
losses = []

for epoch in range(2000):
    z1 = X_xor @ W1 + b1
    a1 = relu(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    
    loss = -np.mean(y_xor * np.log(a2 + 1e-8) + (1 - y_xor) * np.log(1 - a2 + 1e-8))
    losses.append(loss)
    
    m = len(X_xor)
    dz2 = a2 - y_xor
    dW2 = (a1.T @ dz2) / m
    db2 = np.mean(dz2, axis=0, keepdims=True)
    
    da1 = dz2 @ W2.T
    dz1 = da1 * relu_derivative(z1)
    dW1 = (X_xor.T @ dz1) / m
    db1 = np.mean(dz1, axis=0, keepdims=True)
    
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

plt.plot(losses, color='#e74c3c', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss — XOR Problem (2-Layer Network)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nFinal predictions:")
for x, y, pred in zip(X_xor, y_xor, a2):
    print(f"  {x} → {pred[0]:.4f} (target: {y[0]})")

**XOR is the classic test** — a single neuron CANNOT solve it because the classes aren't linearly separable. You need at least one hidden layer, which is why XOR historically proved that multi-layer networks were necessary.

---
## 9. Complete Neural Network Class

Let's package everything into a reusable class. This is what PyTorch does under the hood (with optimizations).

In [ ]:
class NeuralNetwork:
    def __init__(self, layer_sizes):
        self.weights = []
        self.biases = []
        for i in range(len(layer_sizes) - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2.0 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(w)
            self.biases.append(b)
        self.n_layers = len(self.weights)
    
    def forward(self, X):
        self.activations = [X]
        self.z_values = []
        
        for i in range(self.n_layers):
            z = self.activations[-1] @ self.weights[i] + self.biases[i]
            self.z_values.append(z)
            if i == self.n_layers - 1:
                a = sigmoid(z)
            else:
                a = relu(z)
            self.activations.append(a)
        
        return self.activations[-1]
    
    def backward(self, y, lr):
        m = len(y)
        delta = self.activations[-1] - y
        
        for i in range(self.n_layers - 1, -1, -1):
            dW = (self.activations[i].T @ delta) / m
            db = np.mean(delta, axis=0, keepdims=True)
            
            if i > 0:
                delta = (delta @ self.weights[i].T) * relu_derivative(self.z_values[i-1])
            
            self.weights[i] -= lr * dW
            self.biases[i] -= lr * db
    
    def train(self, X, y, lr=0.1, epochs=1000, verbose=True):
        losses = []
        for epoch in range(epochs):
            y_hat = self.forward(X)
            loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))
            losses.append(loss)
            self.backward(y, lr)
            
            if verbose and epoch % (epochs // 5) == 0:
                acc = np.mean((y_hat > 0.5).astype(float) == y) * 100
                print(f"  Epoch {epoch:>5d} | Loss: {loss:.4f} | Accuracy: {acc:.1f}%")
        return losses
    
    def predict(self, X):
        return (self.forward(X) > 0.5).astype(float)

In [ ]:
np.random.seed(42)
nn_xor = NeuralNetwork([2, 8, 1])
losses = nn_xor.train(X_xor, y_xor, lr=0.5, epochs=2000)

def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.forward(grid).reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.8)
    plt.colorbar(label='P(class=1)')
    colors = ['#e74c3c' if yi == 0 else '#2ecc71' for yi in y.ravel()]
    plt.scatter(X[:, 0], X[:, 1], c=colors, s=150, edgecolors='black', zorder=5)
    plt.title(title, fontweight='bold')
    plt.xlabel('x1')
    plt.ylabel('x2')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plt.sca(ax1)
ax1.plot(losses, color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss', fontweight='bold')
ax1.grid(True, alpha=0.3)

plt.sca(ax2)
plot_decision_boundary(nn_xor, X_xor, y_xor, 'XOR Decision Boundary')

plt.tight_layout()
plt.show()

---
## 10. Training on Real-ish Data — Moons Dataset

XOR has only 4 points. Let's train on the moons dataset (200 points, two interleaving half-circles) and watch the decision boundary evolve during training.

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)
y_moons_col = y_moons.reshape(-1, 1).astype(float)

plt.scatter(X_moons[y_moons == 0, 0], X_moons[y_moons == 0, 1],
            c='#e74c3c', label='Class 0', edgecolors='black', s=40)
plt.scatter(X_moons[y_moons == 1, 0], X_moons[y_moons == 1, 1],
            c='#2ecc71', label='Class 1', edgecolors='black', s=40)
plt.title('Moons Dataset', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
np.random.seed(42)
nn_moons = NeuralNetwork([2, 16, 8, 1])

snapshots = [0, 50, 200, 500, 1500, 3000]
snapshot_models = {}
all_losses = []

for epoch in range(3001):
    y_hat = nn_moons.forward(X_moons)
    loss = -np.mean(y_moons_col * np.log(y_hat + 1e-8) + (1 - y_moons_col) * np.log(1 - y_hat + 1e-8))
    all_losses.append(loss)
    nn_moons.backward(y_moons_col, lr=0.5)
    
    if epoch in snapshots:
        snapshot_models[epoch] = {
            'weights': [w.copy() for w in nn_moons.weights],
            'biases': [b.copy() for b in nn_moons.biases]
        }

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

for idx, epoch in enumerate(snapshots):
    snap = snapshot_models[epoch]
    temp_nn = NeuralNetwork([2, 16, 8, 1])
    temp_nn.weights = snap['weights']
    temp_nn.biases = snap['biases']
    Z = temp_nn.forward(grid).reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, levels=50, cmap='RdYlGn', alpha=0.8)
    colors = ['#e74c3c' if yi == 0 else '#2ecc71' for yi in y_moons]
    axes[idx].scatter(X_moons[:, 0], X_moons[:, 1], c=colors, s=15, edgecolors='black', linewidths=0.3)
    axes[idx].set_title(f'Epoch {epoch} (Loss: {all_losses[epoch]:.3f})', fontweight='bold')

plt.suptitle('Decision Boundary Evolution During Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_losses, color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss — Moons Dataset', fontweight='bold')
ax1.grid(True, alpha=0.3)

plt.sca(ax2)
plot_decision_boundary(nn_moons, X_moons, y_moons_col, 'Final Decision Boundary')

y_pred = nn_moons.predict(X_moons)
accuracy = np.mean(y_pred == y_moons_col) * 100
plt.tight_layout()
plt.show()
print(f"Final accuracy: {accuracy:.1f}%")

---
## 11. Training on Circles Dataset

Circles are harder — one class forms a ring around the other. The network needs to learn a radial boundary.

In [ ]:
from sklearn.datasets import make_circles

X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)
y_circles_col = y_circles.reshape(-1, 1).astype(float)

np.random.seed(42)
nn_circles = NeuralNetwork([2, 16, 8, 1])
losses_circles = nn_circles.train(X_circles, y_circles_col, lr=0.5, epochs=3000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(losses_circles, color='#3498db', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss — Circles Dataset', fontweight='bold')
ax1.grid(True, alpha=0.3)

plt.sca(ax2)
plot_decision_boundary(nn_circles, X_circles, y_circles_col, 'Circles Decision Boundary')

accuracy = np.mean(nn_circles.predict(X_circles) == y_circles_col) * 100
plt.tight_layout()
plt.show()
print(f"Final accuracy: {accuracy:.1f}%")

---
## Key Takeaways

1. **A neuron** = weighted sum + bias + activation function
2. **Forward pass** = matrix multiplication at each layer, activation after each
3. **Loss function** = measures prediction error (MSE for regression, cross-entropy for classification)
4. **Backpropagation** = chain rule to compute gradients of loss w.r.t. every weight
5. **Training** = forward → loss → backward → update weights (repeat thousands of times)
6. **More layers** = more complex decision boundaries, but harder to train
7. **Everything PyTorch does** is automating what we just did by hand

Now that you understand the internals, let's move to PyTorch where autograd does the backprop for us.